# Does the TSDF-prior reduce depth error beyond seed noise?

**Question.** When we add a TSDF-prior self-distillation loss on top of RaDe-GS-style training, does the per-view depth error against DTU's ground truth drop by more than the noise we'd see just from changing the seed?

**Design (4 training runs).**

| run              | seed | tsdf λ | purpose |
|---               |---   |---     |---|
| `baseline_s42`   | 42   | —      | reference                           |
| `baseline_s7`    | 7    | —      | noise floor (same model, new seed)  |
| `tsdf_l0.025`    | 42   | 0.025  | small dose of the prior             |
| `tsdf_l0.05`     | 42   | 0.05   | the paper-ish lambda                |

Every run uses `--no-densify` (upstream gsplat 1.5.x DefaultStrategy is broken with our meta dict). Same splat count across all runs, so geometry differences are purely the loss talking.

**Metrics, headline first.**
- `depth_rmse`, `depth_rel_err` against `data/DTU/<scene>/depths/` GT (held-out views only).
- PSNR / SSIM / LPIPS are secondary — we already know they sit inside the seed noise floor.

**Interpretation rule** (printed at the bottom of the comparison cell):
- |Δ(prior − baseline_s42)| > 2 × |baseline_s7 − baseline_s42|  →  **signal**.
- 1× to 2×  →  **borderline**, needs more scenes.
- <1×      →  **noise**.

Total runtime: ~25–35 min on A100, ~2–3 hr on T4.

## 1. Config + GPU

In [ ]:
SCENE         = "scan24"
DATA_FACTOR   = 2
MAX_STEPS     = 30000
DRIVE_DATA    = "/content/drive/MyDrive/mpsplat_data/2DGS_data/dtu.tar.gz"
DRIVE_ROOT    = f"/content/drive/MyDrive/mpsplat_results/{SCENE}"

# The four runs we'll train and re-evaluate.
RUNS = [
    {"name": "baseline_s42", "method": "baseline",   "seed": 42, "tsdf_lambda": 0.0},
    {"name": "baseline_s7",  "method": "baseline",   "seed":  7, "tsdf_lambda": 0.0},
    {"name": "tsdf_l0.025",  "method": "tsdf_prior", "seed": 42, "tsdf_lambda": 0.025},
    {"name": "tsdf_l0.05",   "method": "tsdf_prior", "seed": 42, "tsdf_lambda": 0.05},
]

!nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv

## 2. Clone, install, sanity-check

In [ ]:
import os, shutil, subprocess

if not os.path.isdir("/content/mpsplat"):
    subprocess.run(["git", "clone", "https://github.com/zacsmms/mpsplat.git"], cwd="/content", check=True)
subprocess.run(["git", "pull", "origin", "main"], cwd="/content/mpsplat", check=True)

# Hide the local stripped (Apple-Silicon-only) gsplat so the PyPI CUDA build wins.
local_gsplat = "/content/mpsplat/gsplat"
shadow = "/content/mpsplat/.gsplat_local_stripped"
if os.path.isdir(local_gsplat) and not os.path.isdir(shadow):
    shutil.move(local_gsplat, shadow)
    print("shadowed local stripped gsplat")

!pip uninstall -y mpsplat pycolmap 2>/dev/null | tail -2
!pip install -q gsplat
!pip install -q \
    "git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e" \
    scipy scikit-learn tqdm "torchmetrics[image]" \
    opencv-python tyro Pillow piexif matplotlib imageio

# numpy-2 patch for rmbrualla pycolmap.
import glob
for p in glob.glob("/usr/local/lib/python*/dist-packages/pycolmap/scene_manager.py"):
    subprocess.run(["sed", "-i", r"s/np\.uint64(-1)/np.uint64(np.iinfo(np.uint64).max)/", p], check=True)
print("setup complete")

In [ ]:
import sys, inspect
for k in list(sys.modules):
    if k == "gsplat" or k.startswith("gsplat."):
        del sys.modules[k]
import torch, gsplat
print("torch:",  torch.__version__, "cuda:", torch.cuda.is_available(), torch.version.cuda)
print("gsplat:", gsplat.__version__, "from", gsplat.__file__)
has_extra = "extra_signals" in inspect.signature(gsplat.rasterization).parameters
print("rasterization path:", "single-pass (fork)" if has_extra else "two-pass fallback (upstream)")
assert torch.cuda.is_available(), "No CUDA — change runtime to GPU or reinstall torch with CUDA."

## 3. Drive, dataset, scene path, half-res

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%cd /content/mpsplat
import os, pathlib, glob
from PIL import Image

os.makedirs("data", exist_ok=True)
assert os.path.exists(DRIVE_DATA), f"can't find {DRIVE_DATA}"
if not any(pathlib.Path("data").rglob(SCENE)):
    !tar -xzf "$DRIVE_DATA" -C data/
scene_dirs = list(pathlib.Path("data").rglob(SCENE))
assert scene_dirs, f"{SCENE} not in extracted data"
SCENE_DIR = str(scene_dirs[0])
print("SCENE_DIR =", SCENE_DIR)

src, dst = f"{SCENE_DIR}/images", f"{SCENE_DIR}/images_{DATA_FACTOR}"
if not (os.path.isdir(dst) and len(os.listdir(dst)) == len(os.listdir(src))):
    os.makedirs(dst, exist_ok=True)
    for p in sorted(glob.glob(f"{src}/*")):
        img = Image.open(p)
        img.resize((img.width // DATA_FACTOR, img.height // DATA_FACTOR), Image.LANCZOS) \
           .save(f"{dst}/{os.path.basename(p)}")
    print("wrote", len(os.listdir(dst)), "images to", dst)
else:
    print("images_{} already exists".format(DATA_FACTOR))

assert os.path.isdir(f"{SCENE_DIR}/depths"), "GT depths missing — eval_with_depth.py won't have anything to compare to"
print("GT depths:", len(os.listdir(f'{SCENE_DIR}/depths')), "files")

## 4. Train the four runs

Each run writes to `<DRIVE_ROOT>/<name>/<method>/`. Output is streamed to a log file so the cell stays readable. Total time: ~25 min on A100, ~2 hr on T4.

In [ ]:
import os, subprocess, time
from pathlib import Path

def train(run):
    out = f"{DRIVE_ROOT}/{run['name']}"
    Path(out).mkdir(parents=True, exist_ok=True)
    log = f"{out}/train.log"
    ckpt = f"{out}/{run['method']}/ckpts/ckpt_{MAX_STEPS}.pt"
    if os.path.exists(ckpt):
        print(f"[{run['name']}] already trained — {ckpt}")
        return
    cmd = [
        "python", "examples/experiment_tsdf_prior.py",
        "--data_dir", SCENE_DIR,
        "--result_dir", out,
        "--data_factor", str(DATA_FACTOR),
        "--max_steps", str(MAX_STEPS),
        "--seed", str(run["seed"]),
        "--methods", run["method"],
        "--no-densify",
    ]
    if run["method"] == "tsdf_prior":
        cmd += ["--tsdf_lambda", str(run["tsdf_lambda"])]
    print(f"[{run['name']}] training → {log}")
    t0 = time.time()
    with open(log, "w") as f:
        proc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    print(f"[{run['name']}] done in {(time.time()-t0)/60:.1f} min  (rc={proc.returncode})")

for r in RUNS:
    train(r)
print("\nall training done.")

## 5. Re-evaluate each run against GT depths

`eval_with_depth.py` loads each checkpoint, renders holdout views, and compares to `depths/*.png` from the DTU bundle. Saves `metrics_with_depth.json` plus visual panels under `depth_panels/cmp_*.png`.

In [ ]:
import subprocess

for r in RUNS:
    out = f"{DRIVE_ROOT}/{r['name']}/{r['method']}"
    ckpt = f"{out}/ckpts/ckpt_{MAX_STEPS}.pt"
    print(f"\n=== eval: {r['name']} ===")
    subprocess.run([
        "python", "examples/eval_with_depth.py",
        "--data_dir", SCENE_DIR,
        "--ckpt", ckpt,
        "--out_dir", out,
        "--data_factor", str(DATA_FACTOR),
    ], check=False)

## 6. Headline results — is the prior signal or noise?

In [ ]:
import json, os

def load(r):
    out = f"{DRIVE_ROOT}/{r['name']}/{r['method']}"
    for nm in ("metrics_with_depth.json", "metrics.json"):
        p = f"{out}/{nm}"
        if os.path.exists(p):
            return json.load(open(p))
    return None

data = {r['name']: load(r) for r in RUNS}
for n, m in data.items():
    print(f"{n:14s}  {'(missing)' if m is None else 'ok'}")

def fmt(v, prec=4):
    return f"{v:.{prec}f}" if isinstance(v, float) else ("—" if v is None else str(v))

base = data["baseline_s42"]
noise = data["baseline_s7"]
assert base and noise, "baseline runs missing — re-run the train cell"

print("\n  metric         baseline_s42   baseline_s7    tsdf_l0.025     tsdf_l0.05")
print("  " + "-" * 78)
for k in ["depth_rmse", "depth_rel_err", "psnr", "ssim", "lpips"]:
    bv = base.get(k)
    nv = noise.get(k) if noise else None
    t1 = data["tsdf_l0.025"].get(k) if data["tsdf_l0.025"] else None
    t2 = data["tsdf_l0.05"].get(k)  if data["tsdf_l0.05"]  else None
    print(f"  {k:13s}  {fmt(bv):>11s}    {fmt(nv):>11s}    {fmt(t1):>11s}    {fmt(t2):>11s}")

# ---- Interpretation ----
print()
if all(isinstance(d.get("depth_rmse"), float) for d in data.values() if d):
    noise_floor = abs(noise["depth_rmse"] - base["depth_rmse"])
    print(f"  seed-noise floor on depth_rmse: |s7 − s42| = {noise_floor:.4f} m")
    for label in ["tsdf_l0.025", "tsdf_l0.05"]:
        if data[label] is None:
            continue
        delta = data[label]["depth_rmse"] - base["depth_rmse"]
        ratio = abs(delta) / max(noise_floor, 1e-6)
        direction = "BETTER" if delta < 0 else "WORSE"
        verdict = ("clear signal"  if ratio >= 2.0 else
                   "borderline"   if ratio >= 1.0 else
                   "within noise")
        print(f"  {label:12s}  Δ = {delta:+.4f} m  ({direction}, {ratio:.2f}× noise floor → {verdict})")
else:
    print("  depth_rmse missing for one or more runs — check eval logs.")

## 7. Visual inspection — pred vs. GT depth, per run

Panels are laid out as `[rgb | predicted depth | GT depth | abs error]`, all normalized to the GT depth range so they're directly comparable across runs. Look for:
- speckle / banding on flat regions in the prediction column,
- consistent error patterns across runs (systematic) vs. random differences (noise),
- visibly smoother depth in `tsdf_l0.025` than `baseline_s42` is what you'd see if the prior is doing real work.

In [ ]:
from IPython.display import Image, display, Markdown
import glob

# First three holdout views, all four runs.
view_idxs = [0, 1, 2]
for v in view_idxs:
    display(Markdown(f"### Holdout view {v}"))
    for r in RUNS:
        p = f"{DRIVE_ROOT}/{r['name']}/{r['method']}/depth_panels/cmp_{v:03d}.png"
        if os.path.exists(p):
            display(Markdown(f"**{r['name']}**"))
            display(Image(filename=p))
        else:
            print("missing:", p)

## 8. (Optional) Lambda sweep — find the best prior weight

If §6 showed a clear signal at λ=0.025, sweep finer to find the optimum. Skip otherwise — there's no point optimizing a noise-level effect.

In [ ]:
SWEEP_LAMBDAS = [0.010, 0.025, 0.050, 0.075, 0.100]

for lam in SWEEP_LAMBDAS:
    name = f"sweep_l{lam:g}"
    out  = f"{DRIVE_ROOT}/{name}"
    ckpt = f"{out}/tsdf_prior/ckpts/ckpt_{MAX_STEPS}.pt"
    if os.path.exists(ckpt):
        print(f"[{name}] already trained"); continue
    print(f"\n=== train {name} (λ={lam}) ===")
    !python examples/experiment_tsdf_prior.py \
        --data_dir "$SCENE_DIR" --result_dir "$out" \
        --data_factor $DATA_FACTOR --max_steps $MAX_STEPS \
        --seed 42 --methods tsdf_prior --tsdf_lambda $lam \
        --no-densify 2>&1 | tail -5
    print(f"--- eval {name} ---")
    !python examples/eval_with_depth.py \
        --data_dir "$SCENE_DIR" --ckpt "$ckpt" \
        --out_dir "$out/tsdf_prior" --data_factor $DATA_FACTOR 2>&1 | tail -3

# Print the sweep table.
import json
print("\n  λ        depth_rmse   Δvs s42      psnr")
print("  " + "-"*48)
for lam in SWEEP_LAMBDAS:
    p = f"{DRIVE_ROOT}/sweep_l{lam:g}/tsdf_prior/metrics_with_depth.json"
    if not os.path.exists(p):
        print(f"  {lam:<7g}  (missing)"); continue
    m = json.load(open(p))
    d = m.get("depth_rmse"); psnr = m.get("psnr")
    delta = d - base["depth_rmse"] if isinstance(d, float) else None
    print(f"  {lam:<7g}  {d if d is None else f'{d:.4f}':>10s}   {delta if delta is None else f'{delta:+.4f}':>10s}   {psnr if psnr is None else f'{psnr:.3f}':>7s}")

## 9. What to do next

Based on §6's verdict:

- **clear signal at one or both λ's** → re-run the whole notebook with `SCENE = "scan65"` and `SCENE = "scan114"` to check generalization. Two of three scenes confirming the same direction is much harder to argue with than one.
- **borderline** → re-run §4 with `seed = 999` (third baseline) to tighten the noise estimate, and add `scan65` to see if the borderline becomes a trend.
- **within noise** → the systematic-bias hypothesis from the design discussion holds: TSDF view-fusion averages out *random* per-view error, not bias correlated across views. Pivot to the diagnostic experiment: a synthetic tilted-plane scene where the bias direction is known.

Saved everything to Drive — re-running this notebook on the same `SCENE` skips training/eval where the outputs already exist, so iteration is cheap.